# ARIA-LLM — train YOUR OWN model via knowledge distillation (Colab T4)

This produces **ARIA**: a transformer trained from scratch (own byte-level BPE tokenizer, own random init). **No Qwen weights end up in ARIA.** Qwen is used only in step 4, as a *teacher that writes training data*; ARIA then learns from that text from scratch and never loads Qwen again.

Pipeline: clone → install → Qwen generates a chat dataset → train ARIA from scratch → sanity chat → bundle weights+tokenizer → (optional) upload to Hugging Face for the live demo.

**Before running:** `Runtime → Change runtime type → T4 GPU`, and push your repo to GitHub first (this notebook clones it).

**Honest expectation:** a small from-scratch model gives short, casual replies — genuinely yours, but not GPT-level. Raise `--num` (more data) and `--epochs` for better quality.

In [ ]:
# 1. Confirm GPU (expect Tesla T4)
!nvidia-smi

In [ ]:
# 2. Clone the repo. Change BRANCH if you pushed elsewhere.
REPO = 'https://github.com/Aashutosh2021/ARIA_LLM.git'
BRANCH = 'main'
import os, shutil
if os.path.isdir('ARIA_LLM'):
    shutil.rmtree('ARIA_LLM')
!git clone --branch $BRANCH --depth 1 $REPO
%cd ARIA_LLM

In [ ]:
# 3. Install only what Colab lacks (keep Colab's CUDA torch).
!pip install -q 'transformers>=4.40' 'PyYAML>=6.0' 'tqdm>=4.65' 'huggingface_hub>=0.24'

In [ ]:
# 4. TEACHER STEP: Qwen generates the training dataset (this is the ONLY place
#    Qwen is used). It role-plays ARIA and answers thousands of everyday
#    prompts; a filter drops any answer that leaks 'Qwen/OpenAI/etc' so the
#    student never learns to call itself something else. --num controls dataset
#    size (bigger = better student, slower). Writes data/chat_corpus.txt.
!python scripts/distill_generate.py --num 6000 --device cuda --batch-size 24

In [ ]:
# 5. Peek at the generated data.
!head -n 30 data/chat_corpus.txt

In [ ]:
# 6. STUDENT STEP: train ARIA FROM SCRATCH on the teacher's data. Own BPE
#    tokenizer, random init -- no Qwen weights involved. This is a small model,
#    so it trains fast. More epochs -> more coherent (watch val loss).
!python train_chat.py --device cuda --epochs 30 --vocab-size 8000 \
    --embedding-dim 384 --layers 6 --heads 6 --seq-len 128 --batch-size 64

In [ ]:
# 7. Sanity chat (non-interactive) across several prompts. Replies should be
#    short but on-topic and self-identify as ARIA. This uses ONLY ARIA -- no
#    Qwen loaded.
import torch
from pathlib import Path
from model.gpt import GPT
from training.checkpoint import load_checkpoint
from utils.helper import load_tokenizer
from utils.device import get_device
import torch.nn.functional as F

device = get_device('cuda')
ckpt_path = Path('checkpoints_chat/best.pt')
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
config = ckpt['config']
tok = load_tokenizer(config.get('tokenizer_type', 'bpe'), ckpt_path.parent / 'vocab.json')
config['vocab_size'] = len(tok)
model = GPT.from_config(config); load_checkpoint(ckpt_path, model, map_location=device)
model.to(device).eval()
U, B, END = '<|user|>', '<|bot|>', '<|endoftext|>'

@torch.no_grad()
def reply(q, max_new=60):
    ids = tok.encode(f'{U} {q}\n{B}')
    toks = torch.tensor([ids], device=device); gen = []
    for _ in range(max_new):
        logits, _ = model(toks[:, -config['max_sequence_length']:])
        nl = logits[:, -1, :].float() / 0.8
        k = min(40, nl.size(-1)); th = torch.topk(nl, k, dim=-1).values[..., -1, None]
        nl = nl.masked_fill(nl < th, float('-inf'))
        nt = torch.multinomial(F.softmax(nl, dim=-1), 1)
        gen.append(nt.item()); toks = torch.cat([toks, nt], dim=1)
        t = tok.decode(gen)
        if any(s in t for s in (END, U, B)):
            for s in (END, U, B): t = t.split(s)[0]
            return t.strip()
    return tok.decode(gen).strip()

for q in ['hello, who are you?', 'who made you?', 'what can you do?', 'tell me a fun fact', 'i am bored']:
    print(f'you> {q}\naria> {reply(q)}\n')

In [ ]:
# 8. Bundle weights + tokenizer for deployment -> deploy/aria.pt, deploy/aria_vocab.json
!python scripts/export_for_deploy.py --checkpoint checkpoints_chat/best.pt --out-dir deploy
!ls -la deploy

## 9. Upload to Hugging Face (for the live Space)

Get a **write** token: https://huggingface.co/settings/tokens . Replace `<user>`.

In [ ]:
from huggingface_hub import login, create_repo, upload_file

HF_USER = '<user>'   # <-- your Hugging Face username
REPO_ID = f'{HF_USER}/aria-llm-weights'

login()  # paste write token
create_repo(REPO_ID, repo_type='model', exist_ok=True)
for fname in ['aria.pt', 'aria_vocab.json']:
    upload_file(path_or_fileobj=f'deploy/{fname}', path_in_repo=fname,
                repo_id=REPO_ID, repo_type='model')
    print('uploaded', fname)
print('\nDone. Set HF_MODEL_REPO =', REPO_ID, 'in your Space settings (see DEPLOY.md).')